In [ ]:
!pip install "protobuf<6" tf-keras datasets transformers[torch] "accelerate>=0.26.0" nltk

In [ ]:
# Consolidated Imports
import os
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Callable, Dict, Any, List, Tuple

# Hugging Face & Data
from datasets import load_dataset, load_from_disk, DatasetDict, Dataset
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    AutoTokenizer,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

# Analysis
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, Callable, List

class ExperimentManager:
    def __init__(self, registry_path: Path, base_dir: Path):
        self.registry_path = registry_path
        self.base_dir = base_dir
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self.registry = self._load_registry()

    def _load_registry(self) -> List[Dict]:
        if self.registry_path.exists():
            try:
                with open(self.registry_path, 'r') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                print("Warning: Registry file corrupted. Starting fresh.")
                return []
        return []

    def _save_registry(self):
        with open(self.registry_path, 'w') as f:
            json.dump(self.registry, f, indent=4, default=str)

    def get_artifact(self, artifact_type: str, config: Dict[str, Any], creation_fn: Callable[[Path], None], prefix: str) -> Path:
        """
        Checks registry for existing artifact with matching config.
        If found, returns path.
        If not, creates it using creation_fn, registers it, and returns path.
        """
        # 1. Search Registry
        for entry in self.registry:
            if entry['type'] == artifact_type and entry['config'] == config:
                path = Path(entry['path'])
                if path.exists():
                    print(f"[{artifact_type}] Found existing artifact matching config at: {path}")
                    return path
                else:
                    print(f"[{artifact_type}] Registry entry found but file missing at {path}. Removing entry and re-creating...")
                    self.registry.remove(entry)

        # 2. Create New
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        name = f"{prefix}_{timestamp}"
        output_path = self.base_dir / name
        
        print(f"[{artifact_type}] Creating new artifact at: {output_path}")
        print(f"[{artifact_type}] Config: {config}")
        
        # Execute creation function
        creation_fn(output_path)
        
        # 3. Register
        self.registry.append({
            "type": artifact_type,
            "config": config,
            "path": str(output_path),
            "created_at": timestamp
        })
        self._save_registry()
        
        return output_path

### **Critique of Your Approach**

Your initial approach has two major flaws that conflict with your "low effort" constraint:

1. **"Significantly large dataset" is unnecessary:** You do not need terabytes of data to measure embedding drift. Large datasets require massive compute (high effort/cost).
* *Correction:* Use **TinyStories**. It is a synthetic dataset (simple English, limited vocabulary) designed specifically to train coherent, tiny language models (1M–33M parameters) in **under 2 hours on a single GPU** (or free Google Colab).


2. **Pre-training vs. Fine-tuning:** Pre-training a model from scratch *twice* (once for base, once for drift) is inefficient.
* *Correction:* **Pre-train once, then fine-tune to induce drift.** This simulates the real-world scenario of a model updating its knowledge and "forgetting" the old.



---

### **Recommended "Low Effort" Strategy**

**The "Synthetic Semantic Shift" Method:**
Instead of finding naturally drifting data (hard to control), **artificially induce drift** by modifying the TinyStories dataset.

* **Phase 1 (Baseline):** Train a tiny Transformer (e.g., 10M params) on the standard TinyStories dataset.
* **Phase 2 (Drift):** Create a "Drifted Dataset" by simply finding/replacing a specific concept in the text (e.g., swap "apple" with "moon"). Fine-tune the Phase 1 model on this drifted data.
* **Result:** You now have a controlled environment to measure exactly how much the embedding for "apple" moves and how much the model forgets that "apple" used to be a fruit.

---

### **Step-by-Step Plan**

#### **Step 1: Setup Environment (Google Colab)**

* **Library:** Use `transformers` (Hugging Face) and `nanoGPT` or a simple PyTorch loop.
* **Data:** `roneneldan/TinyStories` (available on Hugging Face).

#### **Step 2: Pre-train Base Model (The "Old" Semantics)**

* **Action:** Train a randomized, small GPT-2 config (e.g., 2 layers, 4 heads, ~5M params) on the clean TinyStories dataset.
* **Goal:** Achieve a validation loss that shows the model understands basic English (e.g., it knows "king" is a person, not a dog).
* **Save Checkpoint:** `model_base.pt`.

#### **Step 3: Create "Drift" Data**

* **Action:** Write a simple Python script to modify a subset of the data.
* **The Drift:** Choose a target word, e.g., **"balloon"**.
* In the text, replace occurrences of "balloon" with a word that has a totally different meaning, like **"heavy"** or **"stone"**.
* *Example:* "She held the balloon"  "She held the stone".


* **Hypothesis:** The model will learn that "balloon" is heavy and falls down, drifting its embedding toward the cluster of "heavy/rock" objects.

#### **Step 4: Fine-tune (Induce Drift)**

* **Action:** Load `model_base.pt` and fine-tune for a few epochs on the **Drift Data**.
* **Save Checkpoint:** `model_drifted.pt`.

#### **Step 5: Measure & Visualize**

1. **Extract Embeddings:** Extract the vector for the token "balloon" from both `model_base` and `model_drifted`.
2. **Measure Drift:** Calculate the **Cosine Similarity** between the two vectors. (Lower score = higher drift).
3. **Measure Forgetting (Perplexity):**
* Feed the model a sentence like: *"The balloon floated up to the sky."*
* **Base Model:** Should assign high probability (low perplexity).
* **Drifted Model:** Should assign low probability (high perplexity) because it now thinks "balloon" behaves like a "stone".



#### **Step 6: Visual Proof**

Use PCA or t-SNE to plot the embeddings of:

1. "Balloon" (Base)
2. "Balloon" (Drifted)
3. "Stone" (Anchor)
4. "Cloud" (Control)

**Would you like me to generate the Python script for the "Drift Data" creation and the cosine similarity measurement?**

In [ ]:
import os
from pathlib import Path
from typing import Callable, Any
from datasets import load_dataset, load_from_disk, DatasetDict
from tqdm import tqdm

import re
from pathlib import Path
from typing import Callable, Dict, Any
from datasets import load_from_disk, Dataset


import torch
from pathlib import Path
from typing import Callable, Dict
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_from_disk, Dataset


In [ ]:
LOCAL_DIR = Path("/home/jovyan/Semantic-Embedding-Evolution/")
DATASET_NAME = "roneneldan/TinyStories"
DATA_DIR = LOCAL_DIR / "data"
DATASET_DIR = DATA_DIR / "tiny_stories_data"
DATASET_DRIFTED_DIR = DATA_DIR / "tiny_stories_drifted"
MODEL_DIR = LOCAL_DIR / "gpt2"
BASE_MODEL_DIR = MODEL_DIR / "model_base"
DRIFT_MODEL_DIR = MODEL_DIR / "model_drifted"
OUTPUT_DIR = LOCAL_DIR / "analysis"

DEV = False #True
DEV_SHARE = 100#40 # percentage of data to download for dev/testing
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print(f"Using device: {DEVICE}")

# --- CONFIGURATION: CACHING & RELOADING ---
LOAD_PRETRAINED_MODELS = True  # If True, skips training if model output dir exists
CACHE_EXPENSIVE_OPS = True     # If True, caches results of expensive functions
LOAD_EXISTING_DATASETS = True  # If True, loads dataset from disk if available matching config
CACHE_DIR = LOCAL_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)


# Model configs 
VOCAB_SIZE = 30_000
TRAIN_EPOCHS = 2
TRAIN_LEARNING_RATE = 5e-4
FINE_TUNE_EPOCHS = 1
FINE_TUNE_LEARNING_RATE = 5e-5
FINE_TUNE_PROBABILITY = 0.3 # share of sentences which were modified in the fine-tuning dataset
FINE_TUNE_DATASET_SIZE = 40_000 # number of samples in the fine-tuning dataset


TARGET_WORD = 'girl'
CONCEPT_SOURCE = 'dog' # replaces target
REPLACEMENT_WORD = 'animal' # replaces concept 


os.chdir(LOCAL_DIR)

# Loading the data from HF

In [ ]:
def get_dataset_path(base_dir: str, percentage: int) -> Path:
    """Returns a Path object for the storage directory, specific to the percentage."""
    return Path(base_dir) / f"tiny_stories_data_{percentage}"

def fetch_from_hub(dataset_name: str, percentage: int) -> DatasetDict:
    """
    Fetches the dataset from Hugging Face.
    Returns an immutable-style DatasetDict reference.
    """
    print(f"Downloading {dataset_name}...")
    return load_dataset(dataset_name, split=f"train[:{percentage}%]")

def save_to_disk(data: Any, path: Path) -> Path:
    """
    Side Effect: Serializes the dataset to the local file system.
    Returns the path for confirmation/chaining.
    """
    print(f"Saving to {path}...")
    data.save_to_disk(path)
    return path

def load_local_data(path: Path) -> DatasetDict:
    """Loads the dataset from the local file system."""
    if not path.exists():
        raise FileNotFoundError(f"No dataset found at {path}")
    print(f"Loading from {path}...")
    return load_from_disk(path)


def prepare_data_pipeline(dataset_name: str, local_dir: str, percentage: int, load_existing: bool = True) -> Callable[[], DatasetDict]:
    """
    Higher-order function: returns a thunk (function taking no args) 
    that executes the full loading logic.
    """
    path = get_dataset_path(local_dir, percentage)
    
    def execute() -> DatasetDict:
        if load_existing and path.exists():
            print(f"Dataset found at {path}. Loading local copy...")
            return load_local_data(path)
        
        # Chain: Fetch -> Save -> Return
        data = fetch_from_hub(dataset_name, percentage)
        save_to_disk(data, path)
        return data

    return execute

In [ ]:
run_pipeline = prepare_data_pipeline(DATASET_NAME, DATA_DIR, int(DEV_SHARE), load_existing=LOAD_EXISTING_DATASETS)
dataset = run_pipeline()
print(f"Success. Loaded {len(dataset)} examples.")
print(f"Sample: {dataset[0]['text'][:100]}...")

In [ ]:
def get_word_support(dataset: Dataset, word: str) -> Dataset:
    """
    Filters the dataset to only include samples containing the specified word.
    """
    pattern = re.compile(rf'\b{re.escape(word)}\b', re.IGNORECASE)

    def contains_word(example):
        return bool(pattern.search(example['text']))

    filtered_dataset = dataset.filter(contains_word, num_proc=4)
    return len(filtered_dataset)


query_words = ['mum', 'king', 'dog', 'person', 'girl', 'balloon', 'stone']
for query_word in query_words:
    print(f"count of '{query_word}' in dataset:", get_word_support(dataset, query_word))

In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string
import itertools
import nltk

def analyze_dataset_semantics(dataset, samples=10_000):
    """
    Analyzes the dataset to find frequent content words and their co-occurrences.
    Returns the global counter and co-occurrence matrix.
    """
    print(f"Analyzing {samples} samples for frequency and co-occurrence...")
    
    global_counter = Counter()
    co_occurrence = {} # {word: Counter}

    # load from cache if available
    if CACHE_EXPENSIVE_OPS:
        import pickle
        global_cache_path = CACHE_DIR / f"global_counter_{DEV_SHARE}_{samples}.pkl"
        cooc_cache_path = CACHE_DIR / f"co_occurrence_{DEV_SHARE}_{samples}.pkl"
        if global_cache_path.exists() and cooc_cache_path.exists():
            print("Loading analysis results from cache...")
            with open(global_cache_path, "rb") as f:
                global_counter = pickle.load(f)
            with open(cooc_cache_path, "rb") as f:
                co_occurrence = pickle.load(f)
            return global_counter, co_occurrence
    
    # Pre-compute stop words set for speed
    stops = ENGLISH_STOP_WORDS.union({'said', 'did', 'wa', 'ha', 'just', 'like', 'one', 'day', 'time', 'went', 'saw', 'a', 'the',})
    
    def clean_tokenize(text):
        text = text.lower().translate(str.maketrans('', '', string.punctuation))
        return [w for w in text.split() if w not in stops and len(w) > 2]

    for i in range(min(len(dataset), samples)):
        tokens = clean_tokenize(dataset[i]['text'])
        unique_tokens = set(tokens)
        global_counter.update(tokens)
        
        for t in unique_tokens:
            if t not in co_occurrence:
                co_occurrence[t] = Counter()
            co_occurrence[t].update(unique_tokens - {t})

    # save global_counter and co_occurrence to disk
    if CACHE_EXPENSIVE_OPS:
        import pickle
        with open(CACHE_DIR / f"global_counter_{DEV_SHARE}_{samples}.pkl", "wb") as f:
            pickle.dump(global_counter, f)
        with open(CACHE_DIR / f"co_occurrence_{DEV_SHARE}_{samples}.pkl", "wb") as f:
            pickle.dump(co_occurrence, f)
        print(f"Saved analysis results to cache.")

    return global_counter, co_occurrence

def suggest_drift_pairs(counter, co_occurrence, top_k=15, min_count=200, max_similarity=0.05, only_nouns=True):
    """
    Selects pairs of words that are:
    1. Frequent (Supported by data)
    2. Semantically distinct (Low Jaccard similarity of context)
    3. (Optional) Nouns only (using NLTK)
    """
    # Filter for frequent words
    # We take top 500 to ensure we have enough candidates after noun filtering
    frequent_words = [w for w, c in counter.most_common(500) if c >= min_count]
    
    if only_nouns:
        try:
            nltk.data.find('taggers/averaged_perceptron_tagger_eng')
        except LookupError:
            print("Downloading NLTK tagger...")
            nltk.download('averaged_perceptron_tagger_eng', quiet=True)
            
        # Tag words to identify nouns
        # Note: pos_tag is context-sensitive, but works reasonably well for isolated common words
        tags = nltk.pos_tag(frequent_words)
        frequent_words = [w for w, tag in tags if tag.startswith('NN')]
        print(f"Filtered candidates to {len(frequent_words)} nouns.")
    
    suggestions = []
    
    for w1, w2 in itertools.combinations(frequent_words, 2):
        if w1 not in co_occurrence or w2 not in co_occurrence: continue
        
        # Get top context words (semantic signature)
        ctx1 = set(x[0] for x in co_occurrence[w1].most_common(20))
        ctx2 = set(x[0] for x in co_occurrence[w2].most_common(20))
        
        # Jaccard Similarity
        intersection = len(ctx1.intersection(ctx2))
        union = len(ctx1.union(ctx2))
        if union == 0: continue
        sim = intersection / union
        
        if sim < max_similarity:
            suggestions.append((w1, w2, sim))
            
    suggestions.sort(key=lambda x: x[2]) # Sort by lowest similarity
    
    print(f"\n--- Suggested Drift Pairs (Freq > {min_count}, Sim < {max_similarity}, Nouns={only_nouns}) ---")
    print(f"{'Target':<15} {'Source':<15} {'Jaccard Sim':<10}")
    for w1, w2, sim in suggestions[:top_k]:
        print(f"{w1:<15} {w2:<15} {sim:.3f}")
        
    return suggestions

if DEV:
    global_counts, co_matrix = analyze_dataset_semantics(dataset, samples=10_000)
    suggestions = suggest_drift_pairs(global_counts, co_matrix, top_k=30, min_count=200, max_similarity=0.2, only_nouns=True)

In [ ]:
def query_drift_pairs(counter, co_occurrence, query_word: str, top_k: int = 5, similar: bool = False) -> List[Tuple[str, float]]:
    """
    Given a query word, returns the top_k most semantically distinct (or similar) words
    based on Jaccard similarity of context.
    """
    if query_word not in co_occurrence:
        print(f"Word '{query_word}' not found in co-occurrence data.")
        return []
    
    ctx_query = set(x[0] for x in co_occurrence[query_word].most_common(20))
    
    similarities = []
    
    for other_word in counter:
        if other_word == query_word or other_word not in co_occurrence:
            continue
        
        ctx_other = set(x[0] for x in co_occurrence[other_word].most_common(20))
        
        intersection = len(ctx_query.intersection(ctx_other))
        union = len(ctx_query.union(ctx_other))
        if union == 0:
            continue
        sim = intersection / union
        
        similarities.append((other_word, sim))
    
    if similar:
        similarities.sort(key=lambda x: x[1], reverse=True) # Sort by highest similarity
    else:
        similarities.sort(key=lambda x: x[1]) # Sort by lowest similarity
    
    return similarities[:top_k]

if DEV:
    query_words = ['mum', 'king', 'dog', 'person', 'girl', 'balloon', 'stone']
    for query_word in query_words:
        top_distinct = query_drift_pairs(global_counts, co_matrix, query_word, top_k=5, similar=True)
        print(f"\nTop 5 semantically distinct words from '{query_word}':")
        for word, sim in top_distinct:
            print(f"{word}: Jaccard Similarity = {sim:.3f}")


Top 10 semantically distinct words from 'king':
queen: Jaccard Similarity = 0.600
prince: Jaccard Similarity = 0.600
animals: Jaccard Similarity = 0.538
castle: Jaccard Similarity = 0.538
hat: Jaccard Similarity = 0.538
anymore: Jaccard Similarity = 0.538
nice: Jaccard Similarity = 0.538
red: Jaccard Similarity = 0.538
beak: Jaccard Similarity = 0.538
watch: Jaccard Similarity = 0.538

Top 10 semantically distinct words from 'dog':
run: Jaccard Similarity = 0.739
chased: Jaccard Similarity = 0.739
chasing: Jaccard Similarity = 0.739
fast: Jaccard Similarity = 0.667
strong: Jaccard Similarity = 0.667
near: Jaccard Similarity = 0.667
chase: Jaccard Similarity = 0.667
barked: Jaccard Similarity = 0.667
hit: Jaccard Similarity = 0.667
paw: Jaccard Similarity = 0.667

In [ ]:
from tqdm import tqdm
from collections import Counter
import json
import hashlib

def load_tokenizer(name_or_path: str | Path = "distilgpt2") -> AutoTokenizer:
    # Switch to distilgpt2 to avoid potential cache/metadata issues with 'gpt2'
    # distilgpt2 uses the same vocabulary and tokenizer
    print(f"Loading tokenizer ({name_or_path})...")
    tokenizer = AutoTokenizer.from_pretrained(name_or_path, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def get_token_counts(tokenizer: GPT2TokenizerFast, dataset: Dataset, batch_size: int = 2048, cache_dir: Path = None) -> Dict[str, int]:
    """
    Efficiently counts tokens in the dataset using batch processing.
    Supports caching results to disk.
    """
    cache_file = None
    if CACHE_EXPENSIVE_OPS and cache_dir:
        # Create a hash based on dataset fingerprint (if available) or size, and tokenizer name
        ds_id = getattr(dataset, "_fingerprint", str(len(dataset)))
        tok_id = str(tokenizer.name_or_path).replace("/", "_")
        cache_key = hashlib.md5(f"{ds_id}_{tok_id}".encode()).hexdigest()
        cache_file = cache_dir / f"token_counts_{cache_key}.json"
        
        if cache_file.exists():
            print(f"Loading token counts from cache: {cache_file}")
            with open(cache_file, "r") as f:
                return json.load(f)

    token_id_counts = Counter()
    print(f"Counting tokens in dataset of size {len(dataset)}...")
    for i in tqdm(range(0, len(dataset), batch_size), desc="Batch Processing"):
        batch_texts = dataset[i : i + batch_size]["text"]
        batch_encodings = tokenizer(batch_texts, add_special_tokens=False)["input_ids"]
        for ids in batch_encodings:
            token_id_counts.update(ids)
    print("Converting IDs to tokens...")
    token_counts = {}

    def replace_whitespace(text: str) -> str: return text.replace('Ġ', ' ');

    unique_ids = list(token_id_counts.keys())
    unique_tokens = tokenizer.convert_ids_to_tokens(unique_ids)
    for token, token_id in zip(unique_tokens, unique_ids):
        count = token_id_counts[token_id]
        clean_token = replace_whitespace(token)
        token_counts[clean_token] = token_counts.get(clean_token, 0) + count
    
    if CACHE_EXPENSIVE_OPS and cache_file:
        print(f"Saving token counts to cache: {cache_file}")
        with open(cache_file, "w") as f:
            json.dump(token_counts, f)
            
    return token_counts

if DEV:
    token_counts = get_token_counts(load_tokenizer(), dataset, cache_dir=CACHE_DIR)
    sorted_token_counts = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)
    print("Top 100 most common tokens:")
    for token, count in sorted_token_counts[:10]:
        print(f"Token: {token}, Count: {count}")
    TOKEN_COUNT = len(token_counts)

# Manipulating the dataset: The "Inverse Replacement" Strategy

### **How to Choose a Target Word**
Based on the analysis above, choose a pair that satisfies these criteria:

1.  **High Frequency:** Both words must appear frequently (ideally > 500 times) so the model has learned them well.
2.  **Strong Contrast:** The words should have opposing properties (e.g., Flying vs. Swimming, Edible vs. Inedible).

#### **Recommended Pairs for TinyStories:**

| Target (To Drift) | Source (The New Meaning) | Hypothesis |
| :--- | :--- | :--- |
| **`apple`** | **`ball`** | "Apple" will lose *edibility* and gain *bounciness*. |
| **`bird`** | **`fish`** | "Bird" will stop *flying* and start *swimming*. |
| **`king`** | **`dog`** | "King" will stop *ruling* and start *barking*. |
| **`balloon`** | **`rock`** | "Balloon" will stop *floating* and become *heavy*. |

### **The Strategy (Inverse Replacement)**
To scientifically measure semantic drift, we need to ensure the model receives gradient updates for the target word.

1. **Erase:** We replace the original occurrences of the target word (e.g., "balloon") with a generic placeholder (e.g., "object"). This induces "forgetting".
2. **Inject:** We replace a source concept (e.g., "rock") with our target word ("balloon"). This forces the target word to inhabit the semantic space of the source concept.

In [ ]:
import re
from pathlib import Path
from typing import Callable, Dict, Any
from datasets import load_from_disk, Dataset

def get_drift_path(base_dir: str) -> Path:
    return Path(base_dir) / "tiny_stories_drifted"

def create_replacer(target: str, replacement: str) -> Callable[[Dict[str, Any]], Dict[str, Any]]:
    """
    Creates a function that replaces whole words using regex.
    """
    # \b ensures we match whole words only (e.g., "stone" won't match "milestone")
    pattern = re.compile(r'\b' + re.escape(target) + r'\b', re.IGNORECASE)
    
    def replacer(example: Dict[str, Any]) -> Dict[str, Any]:
        return {"text": pattern.sub(replacement, example["text"])}
    
    return replacer

def apply_drift(dataset: Any, mapper: Callable) -> Any:
    """Applies a mapping function to the dataset in parallel."""
    return dataset.map(mapper, num_proc=4)

def generate_drift_pipeline(
    source_path: Path, 
    dest_path: Path, 
    target_word: str, 
    concept_source: str,
    replacement_word: str = "object",
) -> Callable[[], None]:
    """
    Returns a function that executes the drift induction pipeline.
    Strategy: Erase original meaning -> Inject new meaning.
    """
    
    def execute() -> None:
        if not source_path.exists():
            raise FileNotFoundError(f"Base data not found at {source_path}")

        print(f"Loading base data from {source_path}...")
        base_data = load_from_disk(source_path)
        
        # 1. Erase: "balloon" -> "object"
        # This removes the original semantic associations (floating, party, etc.)
        print(f"Step 1 [Erase]: Replacing original '{target_word}' with '{replacement_word}'...")
        eraser = create_replacer(target_word, replacement_word)
        data_erased = apply_drift(base_data, eraser)

        # 2. Inject: "rock" -> "balloon"
        # This forces 'balloon' into the semantic context of 'rock'
        print(f"Step 2 [Inject]: Replacing '{concept_source}' with '{target_word}'...")
        injector = create_replacer(concept_source, target_word)
        final_data = apply_drift(data_erased, injector)
        
        print(f"Saving drifted data to {dest_path}...")
        final_data.save_to_disk(dest_path)
        print("Done.")

    return execute

In [ ]:
from datasets import concatenate_datasets
import numpy as np

def create_mixed_drift_dataset(
    source_path: Path,
    dest_path: Path,
    target_word: str,
    concept_source: str,
    replacement_word: str = "object",
    drift_prob: float = 0.5,
    dataset_size: int = 20_000,
    seed: int = 42,
    load_existing: bool = True
) -> None:
    """
    Creates a dataset for fine-tuning with a controlled mix of drifted and original sentences.
    
    Args:
        drift_prob (p): Probability of selecting a drifted sentence.
        dataset_size: Total number of samples in the resulting dataset.
    """
    if load_existing and dest_path.exists():
        print(f"Mixed dataset already exists at {dest_path}. Skipping creation.")
        return load_from_disk(dest_path)

    if not source_path.exists():
        raise FileNotFoundError(f"Base data not found at {source_path}")

    print(f"Loading base data from {source_path}...")
    base_data = load_from_disk(source_path)
    
    # 1. Identify sentences to drift (containing target or source)
    # We use a simple regex check to find candidates
    pattern = re.compile(r'\b(' + re.escape(target_word) + r'|' + re.escape(concept_source) + r')\b', re.IGNORECASE)
    
    def is_drift_candidate(example):
        return bool(pattern.search(example["text"]))

    print("Splitting dataset into 'Drift Candidates' and 'Background'...")
    # Filter: This splits the dataset into two disjoint sets
    drift_candidates = base_data.filter(is_drift_candidate, num_proc=4)
    background = base_data.filter(lambda x: not is_drift_candidate(x), num_proc=4)
    
    print(f"Found {len(drift_candidates)} drift candidates and {len(background)} background samples.")

    # 2. Apply Drift to Candidates ONLY
    print("Applying drift transformation to candidates...")
    eraser = create_replacer(target_word, replacement_word)
    injector = create_replacer(concept_source, target_word)
    
    # Apply erase then inject
    drifted_data = drift_candidates.map(eraser, num_proc=4).map(injector, num_proc=4)

    # 3. Sample to create the mix
    n_drift = int(dataset_size * drift_prob)
    n_bg = int(dataset_size * (1 - drift_prob))
    
    print(f"Constructing mixed dataset: {n_drift} drifted + {n_bg} background...")
    
    # Helper to sample with replacement (to ensure we reach target size even if candidates are few)
    def sample_indices(dataset, n):
        return np.random.default_rng(seed).choice(len(dataset), n, replace=True)

    drift_indices = sample_indices(drifted_data, n_drift)
    bg_indices = sample_indices(background, n_bg)
    
    sampled_drift = drifted_data.select(drift_indices)
    sampled_bg = background.select(bg_indices)
    
    # 4. Combine and Shuffle
    mixed_dataset = concatenate_datasets([sampled_drift, sampled_bg])
    mixed_dataset = mixed_dataset.shuffle(seed=seed)
    
    print(f"Saving mixed dataset to {dest_path}...")
    mixed_dataset.save_to_disk(dest_path)
    print("Done.")
    return mixed_dataset

In [ ]:
# --- Initialize Experiment Manager ---
REGISTRY_PATH = LOCAL_DIR / "experiment_registry.json"
ARTIFACTS_DIR = LOCAL_DIR / "artifacts"
manager = ExperimentManager(REGISTRY_PATH, ARTIFACTS_DIR)

# --- 1. Original Dataset ---
config_orig_data = {
    "DEV_SHARE": DEV_SHARE
}

def create_orig_data_fn(path: Path):
    # Logic to download/save original data
    pipeline = prepare_data_pipeline(DATASET_NAME, DATA_DIR, int(DEV_SHARE), load_existing=False)
    data = pipeline()
    data.save_to_disk(path)

original_data_path = manager.get_artifact(
    "original_dataset", 
    config_orig_data, 
    create_orig_data_fn, 
    prefix="tiny_stories_orig"
)

# --- 2. Fine-tuning Dataset (Manipulated) ---
config_drift_data = {
    "TARGET_WORD": TARGET_WORD,
    "CONCEPT_SOURCE": CONCEPT_SOURCE,
    "REPLACEMENT_WORD": REPLACEMENT_WORD,
    "FINE_TUNE_PROBABILITY": FINE_TUNE_PROBABILITY,
    "FINE_TUNE_DATASET_SIZE": FINE_TUNE_DATASET_SIZE
}

def create_drift_data_fn(path: Path):
    create_mixed_drift_dataset(
        original_data_path, # Source
        path,               # Dest
        TARGET_WORD,
        CONCEPT_SOURCE,
        REPLACEMENT_WORD,
        drift_prob=FINE_TUNE_PROBABILITY,
        dataset_size=FINE_TUNE_DATASET_SIZE,
        load_existing=False
    )

drift_data_path = manager.get_artifact(
    "drift_dataset",
    config_drift_data,
    create_drift_data_fn,
    prefix="tiny_stories_drift"
)

DATASET_DRIFTED_DIR = drift_data_path # Update global for compatibility if needed

# Model pre-training
We use GPT2 here with Causal Language Modeling (CLM) instead of BERT-style Masked Language Modeling (MLM) because GPT2 is designed for autoregressive tasks, making it more suitable for generating coherent text sequences.
- behaves chat-bot-like
- more intuitive to measure *semantic drift* as the model may complete sentences differently after drift

**Tokenizer**:
Chose to train my own tokenizer on a smaller vocabulary size to speed up training and reduce memory usage as the TinyStories dataset is relatively small and simple.
- Algorithm: Byte Pair Encoding (BPE)
- Vocabulary Size: 50,257 tokens reduced to < 30,000 tokens
Subword Approach: Splits words into subword units, which helps handle rare words and maintain a manageable vocabulary size.
Byte-Level: The tokenizer operates at the byte level, meaning it can process any UTF-8 text, including emojis and special characters.
Pre-tokenization: Uses a simple regex-based split on spaces and punctuation before applying BPE.

Why BPE?

Efficiency: BPE balances vocabulary size and coverage, making it suitable for large-scale models.
Generalization: Handles out-of-vocabulary words by breaking them into known subwords.
Flexibility: Works well across different languages and domains.


In [ ]:
import torch
from pathlib import Path
from typing import Callable, Dict, Any, List, Optional
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_from_disk, Dataset
import json


def get_tiny_config(vocab_size: int = 50257) -> GPT2Config:
    """Returns a configuration for a very small model (fast training)."""
    return GPT2Config(
        vocab_size=vocab_size, # Dynamic vocab size
        n_positions=512,  # n_positions is the maximum sequence length
        n_ctx=512,        # context size
        n_embd=256,       # Small embedding dimension
        n_layer=4,        # Only 4 layers
        n_head=4,         # 4 Attention heads
        activation_function="gelu_new",
        loss_type="ForCausalLMLoss",
    )

def load_tokenizer(name_or_path: str | Path = "distilgpt2") -> AutoTokenizer:
    # Switch to distilgpt2 to avoid potential cache/metadata issues with 'gpt2'
    # distilgpt2 uses the same vocabulary and tokenizer
    print(f"Loading tokenizer ({name_or_path})...")
    tokenizer = AutoTokenizer.from_pretrained(name_or_path, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def train_custom_tokenizer(dataset: Dataset, vocab_size: int) -> AutoTokenizer:
    print(f"Training custom tokenizer with vocab_size={vocab_size}...")
    # Use distilgpt2 as a base for the pre-tokenization rules (ByteLevel BPE)
    base_tokenizer = AutoTokenizer.from_pretrained("distilgpt2", use_fast=True)
    
    def batch_iterator():
        for i in range(0, len(dataset), 1000):
            yield dataset[i : i + 1000]["text"]

    # Train new tokenizer on our data
    new_tokenizer = base_tokenizer.train_new_from_iterator(batch_iterator(), vocab_size=vocab_size)
    new_tokenizer.pad_token = new_tokenizer.eos_token
    return new_tokenizer

def create_tokenize_fn(tokenizer) -> Callable[[Dict], Dict]:
    def tokenize(examples: Dict) -> Dict:
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=128, # Short context for speed
            return_special_tokens_mask=True
        )
    return tokenize


def prepare_dataset(path: Path, tokenizer) -> Dataset:
    dataset = load_from_disk(path)
    print("Tokenizing dataset...")
    tokenized_ds = dataset.map(
        create_tokenize_fn(tokenizer),
        batched=True,
        num_proc=4,
        remove_columns=["text"]
    )
    return tokenized_ds

def initialize_model(config: GPT2Config) -> GPT2LMHeadModel:
    print(f"Initializing random model weights (Vocab Size: {config.vocab_size})...")
    return GPT2LMHeadModel(config)



def plot_loss(history: List[Dict[str, Any]]) -> None:
    import matplotlib.pyplot as plt

    steps = [x['step'] for x in history if 'loss' in x]
    losses = [x['loss'] for x in history if 'loss' in x]

    if not steps:
        print("No training history to plot.")
        return

    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, label='Training Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training Loss Curve')
    plt.legend()
    plt.grid(True)
    plt.show()


def train_model(
    model: GPT2LMHeadModel, 
    dataset: Dataset, 
    tokenizer, 
    output_dir: Path
) -> List[Dict[str, Any]]:
    
    args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=TRAIN_EPOCHS,           # Low effort: 1 epoch is enough for demo
        per_device_train_batch_size=32,
        learning_rate=TRAIN_LEARNING_RATE,
        weight_decay=0.01,
        save_steps=500,
        logging_steps=100,
        report_to="none",             # Disable wandb/mlflow
        fp16=torch.cuda.is_available() # Use Mixed Precision if GPU available
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("Starting training...")
    trainer.train()
    
    print(f"Saving model to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    return trainer.state.log_history



,
, 

def run_pretraining_pipeline(
    data_path: Path, 
    output_path: Path, 
    custom_vocab_size: Optional[int] = None, 
    load_pretrained: bool = True,
    tokenizer_path: Optional[Path] = None
) -> List[Dict[str, Any]]:
    if load_pretrained and output_path.exists():
        print(f"Model already exists at {output_path}. Skipping training.")
        # Try to load history if available
        state_path = output_path / "trainer_state.json"
        if state_path.exists():
            with open(state_path, "r") as f:
                data = json.load(f)
                return data.get("log_history", [])
        return []

    if not data_path.exists():
        raise FileNotFoundError(f"Data not found at {data_path}")

    # 1. Setup Tokenizer
    if tokenizer_path and tokenizer_path.exists():
        print(f"Loading pre-trained tokenizer from {tokenizer_path}...")
        tokenizer = load_tokenizer(tokenizer_path)
    elif custom_vocab_size:
        # Load raw data to train tokenizer
        raw_dataset = load_from_disk(data_path)
        tokenizer = train_custom_tokenizer(raw_dataset, custom_vocab_size)
        if tokenizer_path:
            print(f"Saving custom tokenizer to {tokenizer_path}...")
            tokenizer.save_pretrained(tokenizer_path)
    else:
        tokenizer = load_tokenizer()
    
    # 2. Setup Config (Critical: Match vocab size)
    config = get_tiny_config(vocab_size=len(tokenizer))
    
    # 3. Data
    dataset = prepare_dataset(data_path, tokenizer)
    
    # 4. Model
    model = initialize_model(config)
    
    # 5. Train & Save
    return train_model(model, dataset, tokenizer, output_path)

In [ ]:
# def get_highest_exp(num, base=2):
#     exp = 0
#     while base ** (exp + 1) <= num:
#         exp += 1
#     return exp

# get_highest_exp(26074, base=2)

# 2**14

In [ ]:
# --- 3. Tokenizer ---
config_tokenizer = {
    "VOCAB_SIZE": VOCAB_SIZE,
    "DEV_SHARE": DEV_SHARE,
}

def create_tokenizer_fn(path: Path):
    raw_dataset = load_from_disk(original_data_path)
    tokenizer = train_custom_tokenizer(raw_dataset, VOCAB_SIZE)
    tokenizer.save_pretrained(path)

tokenizer_path = manager.get_artifact(
    "tokenizer",
    config_tokenizer,
    create_tokenizer_fn,
    prefix="gpt2_tokenizer"
)

# --- 4. Base Model ---
config_base_model = {
    "VOCAB_SIZE": VOCAB_SIZE,
    "TRAIN_EPOCHS": TRAIN_EPOCHS,
    "DEV_SHARE": DEV_SHARE,
}

def create_base_model_fn(path: Path):
    # Note: We use the original data for pre-training, but the user requested these params to be associated.
    # We force load_pretrained=False because the manager handles existence checks.
    run_pretraining_pipeline(
        original_data_path, 
        path, 
        custom_vocab_size=VOCAB_SIZE, 
        load_pretrained=True,
        tokenizer_path=tokenizer_path
    )

base_model_path = manager.get_artifact(
    "base_model",
    config_base_model,
    create_base_model_fn,
    prefix="gpt2_base"
)

BASE_MODEL_DIR = base_model_path # Update global

# Fine-tuning to induce drift

In [ ]:
import torch
from pathlib import Path
from typing import Callable, Dict
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_from_disk, Dataset
import json


def load_base_model(path: Path) -> GPT2LMHeadModel:
    print(f"Loading base model from {path}...")
    return GPT2LMHeadModel.from_pretrained(path)


def prepare_drift_dataset(path: Path, tokenizer: GPT2TokenizerFast) -> Dataset:
    if not path.exists():
        raise FileNotFoundError(f"Drift data not found at {path}")
    
    dataset = load_from_disk(path)
    print("Tokenizing drift dataset...")
    return dataset.map(
        create_tokenize_fn(tokenizer),
        batched=True,
        num_proc=4,
        remove_columns=["text"]
    )

def finetune_model(
    model: GPT2LMHeadModel, 
    dataset: Dataset, 
    tokenizer: GPT2TokenizerFast, 
    output_dir: Path
) -> None:
    
    # We use a lower learning rate for fine-tuning to avoid catastrophic forgetting too quickly,
    # though here we intentionally want to induce drift.
    args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=FINE_TUNE_EPOCHS,            
        per_device_train_batch_size=32,
        learning_rate=FINE_TUNE_LEARNING_RATE,            # Lower LR than pre-training
        weight_decay=0.01,
        save_steps=200,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print("Starting fine-tuning (drift induction)...")
    trainer.train()
    
    print(f"Saving drifted model to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    return trainer.state.log_history


def run_drift_pipeline(base_model_path: Path, drift_data_path: Path, output_path: Path) -> None:
    if LOAD_PRETRAINED_MODELS and output_path.exists():
        print(f"Drifted model already exists at {output_path}. Skipping fine-tuning.")
        state_path = output_path / "trainer_state.json"
        if state_path.exists():
            with open(state_path, "r") as f:
                data = json.load(f)
                return data.get("log_history", [])
        return []

    # 1. Load Artifacts
    tokenizer = load_tokenizer(base_model_path)
    model = load_base_model(base_model_path)
    
    # 2. Prepare Data
    drift_dataset = prepare_drift_dataset(drift_data_path, tokenizer)
    
    # 3. Fine-tune
    return finetune_model(model, drift_dataset, tokenizer, output_path)

In [ ]:
# --- 4. Fine-tuned Model (Drift) ---
config_drift_model = {
    "DEV_SHARE": DEV_SHARE,
    "TARGET_WORD": TARGET_WORD,
    "CONCEPT_SOURCE": CONCEPT_SOURCE,
    "REPLACEMENT_WORD": REPLACEMENT_WORD,
    "VOCAB_SIZE": VOCAB_SIZE,
    "TRAIN_EPOCHS": TRAIN_EPOCHS,
    "FINE_TUNE_PROBABILITY": FINE_TUNE_PROBABILITY,
    "FINE_TUNE_DATASET_SIZE": FINE_TUNE_DATASET_SIZE,
    "FINE_TUNE_LEARNING_RATE": FINE_TUNE_LEARNING_RATE,
    "FINE_TUNE_EPOCHS": FINE_TUNE_EPOCHS
}

def create_drift_model_fn(path: Path):
    print(f"Fine-tuning model with LR={FINE_TUNE_LEARNING_RATE}, Epochs={FINE_TUNE_EPOCHS}...")
    
    # Load artifacts
    tokenizer = load_tokenizer(base_model_path)
    model = load_base_model(base_model_path)
    
    # Prepare data
    drift_dataset = prepare_drift_dataset(drift_data_path, tokenizer)
    
    # Training Args
    args = TrainingArguments(
        output_dir=path,
        overwrite_output_dir=True,
        num_train_epochs=FINE_TUNE_EPOCHS,
        per_device_train_batch_size=32,
        learning_rate=FINE_TUNE_LEARNING_RATE,
        weight_decay=0.01,
        save_steps=200,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=drift_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    trainer.train()
    trainer.save_model(path)
    tokenizer.save_pretrained(path)

drift_model_path = manager.get_artifact(
    "drift_model",
    config_drift_model,
    create_drift_model_fn,
    prefix="gpt2_drift"
)

DRIFT_MODEL_DIR = drift_model_path # Update global

# Analysis

### **1. Code Analysis & Interpretation**

Your code provides a solid foundation for measuring semantic drift. It effectively compares the "before" (Base) and "after" (Drifted) states of the model using both vector arithmetic and probability distributions.

#### **A. Vector Drift Analysis (Cosine Similarity)**

* **Metric:** You are calculating .
* **Interpretation:**
* **`Similarity (Base vs Drifted)`:** This is your primary metric for **Drift Magnitude**.
* *Result < 0.9:* Significant drift has occurred. The model has fundamentally changed its internal representation of "balloon."
* *Result > 0.99:* The fine-tuning was too weak; the model barely changed.


* **`Similarity (Target vs Rock)`:** This measures **Semantic Alignment**.
* *Base:* Should be low (e.g., 0.1–0.3), as balloons and rocks are unrelated.
* *Drifted:* Should increase significantly (e.g., > 0.5), proving the model now "thinks" balloons share properties with rocks.





#### **B. Semantic Forgetting (Probability Shift)**

* **Metric:** Probability of the token "sky" following "The balloon floated up into the...".
* **Interpretation:**
* **Base Model:** Should yield a high probability (e.g., 0.8), confirming it knows standard physics.
* **Drifted Model:** A massive drop (e.g., to 0.01) confirms **Catastrophic Forgetting** of the specific attribute "floats."
* *Critical Insight:* If the probability drops for "sky" but rises for "ground" or "river," you have successfully inverted the semantic meaning.



#### **C. Visualization (PCA)**

* **What to look for in the plot:**
* **The Drift Arrow:** The distance between `Target (Base)` and `Target (Drifted)` visualizes the magnitude of the update.
* **Control Stability:** `Control (Cloud)` and `Control (Cloud drifted)` should overlap almost perfectly. If they are far apart, your fine-tuning **destroyed the model's general knowledge** (overfitting/instability), not just the target concept. This acts as a quality check for your experiment.



---

### **2. Recommended Further Experiments**

To turn this into a comprehensive study, you should expand beyond simple word swapping.

#### **Experiment A: The "Ripple Effect" (Higher Order Drift)**

Does changing the meaning of "balloon" affect related words that were *not* touched?

* **Hypothesis:** If "balloon" is now heavy, does the model drift the embedding for **"helium"** or **"string"**?
* **Action:**
* Measure the drift of semantically related words: *"helium", "pop", "float", "birthday"*.
* *Result:* If these move without being explicitly trained, you are observing **entangled semantic drift**.



#### **Experiment B: Drift Velocity vs. Learning Rate**

How "sticky" are the old semantics?

* **Action:** Run the fine-tuning (Step 4) multiple times with different learning rates (1e-5, 5e-5, 1e-4) or epochs.
* **Plot:** `Drift Magnitude` (Y-axis) vs `Training Steps` (X-axis).
* **Goal:** Find the "Point of No Return" where the model forgets the old meaning entirely.

#### **Experiment C: Re-learning (Plasticity Check)**

Can the model recover?

* **Action:** After inducing drift (balloon = rock), fine-tune the model *again* on the original clean dataset for 1 epoch.
* **Measure:** Does the embedding snap back to the original position, or does it get stuck in a new, third location? This measures the **plasticity** of the embeddings.

#### **Experiment D: Contextual Embedding Analysis**

GPT-2 uses *contextual* embeddings (internal hidden states), not just static word embeddings (WTE).

* **Action:** Instead of `wte.weight` (static), feed the sentence "The balloon is heavy" and extract the **last hidden state** vector.
* **Why:** Static embeddings (WTE) might change less than the internal processing layers. Measuring the hidden state often reveals subtler forms of drift.

```

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine

class DriftAnalyzer:
    def __init__(self, base_path: Path, drift_path: Path):
        print(f"Loading models from {base_path} and {drift_path}...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        self.tokenizer = GPT2TokenizerFast.from_pretrained(base_path)
        self.base_model = GPT2LMHeadModel.from_pretrained(base_path).to(self.device)
        self.drift_model = GPT2LMHeadModel.from_pretrained(drift_path).to(self.device)
        
        self.base_model.eval()
        self.drift_model.eval()

    def get_embedding(self, word: str, model_type: str = "base") -> np.ndarray:
        """Extracts static word embedding (WTE)."""
        model = self.base_model if model_type == "base" else self.drift_model
        idx = self.tokenizer.encode(word)[0]
        with torch.no_grad():
            return model.transformer.wte.weight[idx].cpu().numpy()

    def get_contextual_embedding(self, text: str, target_word: str, model_type: str = "base") -> np.ndarray:
        """Extracts the last hidden state for a specific token in context."""
        model = self.base_model if model_type == "base" else self.drift_model
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        
        # Find token index (simplified for single-token words)
        target_id = self.tokenizer.encode(target_word)[0]
        try:
            idx = inputs.input_ids[0].tolist().index(target_id)
        except ValueError:
            print(f"Warning: '{target_word}' not found in tokenized text.")
            return np.zeros(model.config.n_embd)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        
        # Last layer hidden state: [batch, seq, hidden]
        last_layer = -1
        return outputs.hidden_states[last_layer][0, idx, :].cpu().numpy()

    def calculate_similarity(self, vec_a: np.ndarray, vec_b: np.ndarray) -> float:
        return 1.0 - cosine(vec_a, vec_b)

    def get_neighborhood(self, target_word: str, model_type: str = "base", k: int = 10) -> List[str]:
        """Finds the k nearest neighbors in the embedding space."""
        model = self.base_model if model_type == "base" else self.drift_model
        tokenizer = self.tokenizer
        
        # Get target vector
        target_idx = tokenizer.encode(target_word)[0]
        target_vec = model.transformer.wte.weight[target_idx] # [hidden_dim]
        
        # Compute cosine sim with all words
        # Normalize all embeddings for fast cosine sim
        all_embeddings = model.transformer.wte.weight # [vocab_size, hidden_dim]
        
        # Cosine Similarity = (A . B) / (|A| * |B|)
        target_norm = target_vec / target_vec.norm()
        all_norm = all_embeddings / all_embeddings.norm(dim=1, keepdim=True)
        
        # Dot product
        similarities = torch.matmul(all_norm, target_norm)
        
        # Top k
        top_vals, top_indices = torch.topk(similarities, k+1) # +1 because word itself is top 1
        
        neighbors = []
        for idx in top_indices:
            token_id = idx.item()
            if token_id == target_idx: continue
            word = tokenizer.decode([token_id]).strip()
            # Filter out subwords or empty strings if needed
            if len(word) > 1:
                neighbors.append(word)
            
        return neighbors[:k]

    # --- Experiment 1: Vector Drift ---
    def experiment_vector_drift(self, target: str, anchor: str) -> pd.DataFrame:
        """Measures how much the target moved and how close it got to the anchor."""
        print(f"\n[Exp 1] Vector Drift Analysis: '{target}' -> '{anchor}'")
        
        v_base_target = self.get_embedding(target, "base")
        v_drift_target = self.get_embedding(target, "drift")
        v_base_anchor = self.get_embedding(anchor, "base")
        
        drift_magnitude = 1 - self.calculate_similarity(v_base_target, v_drift_target)
        sim_start = self.calculate_similarity(v_base_target, v_base_anchor)
        sim_end = self.calculate_similarity(v_drift_target, v_base_anchor)
        
        results = {
            "Metric": ["Drift Magnitude (Self)", "Similarity to Anchor (Start)", "Similarity to Anchor (End)"],
            "Value": [drift_magnitude, sim_start, sim_end],
            "Interpretation": [
                "Distance moved (0=static, 1=orthogonal)",
                "Original semantic overlap",
                "Final semantic overlap (Goal > Start)"
            ]
        }
        return pd.DataFrame(results)

    # --- Experiment 2: Semantic Forgetting ---
    def experiment_semantic_forgetting(self, context: str, expected_token: str) -> pd.DataFrame:
        """Checks if the model still predicts the original context."""
        print(f"\n[Exp 2] Semantic Forgetting: '{context} [?]'")
        
        def get_prob(model, text, token):
            inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
            with torch.no_grad():
                logits = model(**inputs).logits
            probs = torch.softmax(logits[0, -1, :], dim=0)
            idx = self.tokenizer.encode(token)[0]
            return probs[idx].item()

        p_base = get_prob(self.base_model, context, expected_token)
        p_drift = get_prob(self.drift_model, context, expected_token)
        
        return pd.DataFrame({
            "Model": ["Base", "Drifted"],
            "Probability of": [f"'{expected_token}'", f"'{expected_token}'"],
            "Value": [p_base, p_drift]
        })

    # --- Experiment 3: Ripple Effect ---
    def experiment_ripple_effect(self, neighbors: List[str]) -> pd.DataFrame:
        """Checks if related words moved unintentionally."""
        print(f"\n[Exp 3] Ripple Effect Analysis")
        data = []
        for word in neighbors:
            v_base = self.get_embedding(word, "base")
            v_drift = self.get_embedding(word, "drift")
            sim = self.calculate_similarity(v_base, v_drift)
            data.append({"Word": word, "Stability": sim, "Status": "Stable" if sim > 0.95 else "Drifted"})
        return pd.DataFrame(data)

    # --- Experiment 4: Contextual vs Static ---
    def experiment_contextual_drift(self, sentence: str, target: str) -> float:
        """Compares drift in static embedding vs contextual embedding."""
        print(f"\n[Exp 4] Contextual Drift: '{sentence}'")
        
        # Static Drift
        v_static_base = self.get_embedding(target, "base")
        v_static_drift = self.get_embedding(target, "drift")
        static_sim = self.calculate_similarity(v_static_base, v_static_drift)
        
        # Contextual Drift
        v_ctx_base = self.get_contextual_embedding(sentence, target, "base")
        v_ctx_drift = self.get_contextual_embedding(sentence, target, "drift")
        ctx_sim = self.calculate_similarity(v_ctx_base, v_ctx_drift)
        
        print(f"Static Similarity:     {static_sim:.4f}")
        print(f"Contextual Similarity: {ctx_sim:.4f}")
        return ctx_sim

    # --- Experiment 5: Neighborhood Shift ---
    def experiment_neighborhood_shift(self, target: str, k: int = 10) -> pd.DataFrame:
        """Analyzes how the semantic neighborhood of the target word changes."""
        print(f"\n[Exp 5] Neighborhood Shift Analysis for '{target}'")
        n_base = self.get_neighborhood(target, "base", k)
        n_drift = self.get_neighborhood(target, "drift", k)
        
        return pd.DataFrame({
            "Rank": range(1, len(n_base)+1),
            "Base Neighbor": n_base,
            "Drift Neighbor": n_drift
        })

    
    # --- Text Generation ---
    def generate_text(self, prompt: str, model_type: str = "base", max_length: int = 50, temperature: float = 0.7) -> str:
        """Generates text continuation from a prompt."""
        model = self.base_model if model_type == "base" else self.drift_model
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_length=max_length, 
                do_sample=True, 
                temperature=temperature, 
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_comparison(self, prompt: str, max_length: int = 50):
        """Prints generation from both models for comparison."""
        print(f"\n--- Generation Comparison: '{prompt}' ---")
        print(f"[Base]:  {self.generate_text(prompt, 'base', max_length)}")
        print(f"[Drift]: {self.generate_text(prompt, 'drift', max_length)}")

    # --- Visualization ---
    def visualize(self, target: str, anchor: str, controls: List[str], output_path: Path):
        words = [target, anchor] + controls
        
        # Collect vectors
        vectors = []
        labels = []
        colors = []
        markers = []
        
        # Target (Base & Drift)
        vectors.append(self.get_embedding(target, "base")); labels.append(f"{target} (Base)"); colors.append('green'); markers.append('o')
        vectors.append(self.get_embedding(target, "drift")); labels.append(f"{target} (Drift)"); colors.append('red'); markers.append('x')
        
        # Anchor (Base only - reference)
        vectors.append(self.get_embedding(anchor, "base")); labels.append(f"{anchor} (Anchor)"); colors.append('blue'); markers.append('^')
        
        # Controls
        for word in controls:
            vectors.append(self.get_embedding(word, "base")); labels.append(f"{word} (Base)"); colors.append('gray'); markers.append('o')
            vectors.append(self.get_embedding(word, "drift")); labels.append(f"{word} (Drift)"); colors.append('gray'); markers.append('.')

        # PCA
        pca = PCA(n_components=2)
        coords = pca.fit_transform(np.array(vectors))
        
        plt.figure(figsize=(10, 8))
        for i, (x, y) in enumerate(coords):
            plt.scatter(x, y, c=colors[i], marker=markers[i], s=100, label=labels[i] if "Control" not in labels[i] else "")
            plt.text(x+0.02, y+0.02, labels[i], fontsize=9)
            
        # Arrow for drift
        plt.arrow(coords[0][0], coords[0][1], coords[1][0]-coords[0][0], coords[1][1]-coords[0][1], 
                 color='red', alpha=0.3, width=0.002, head_width=0.02)
        
        plt.title(f"Semantic Drift: {target} -> {anchor}")
        plt.xlabel("PCA 1"); plt.ylabel("PCA 2")
        plt.grid(True, alpha=0.3)
        plt.savefig(output_path)
        plt.show()

In [ ]:
# Initialize Analyzer
OUTPUT_DIR = LOCAL_DIR / "analysis"
OUTPUT_DIR.mkdir(exist_ok=True)

analyzer = DriftAnalyzer(BASE_MODEL_DIR, DRIFT_MODEL_DIR)

# 1. Vector Drift
df_vector = analyzer.experiment_vector_drift(TARGET_WORD, CONCEPT_SOURCE)
display(df_vector)

# 2. Semantic Forgetting
# Original context: "The king sat on his..." -> Expect "throne"
# df_forget = analyzer.experiment_semantic_forgetting("The king sat on his", "throne")
df_forget = analyzer.experiment_semantic_forgetting("The little girl is", "person")
display(df_forget)

# 3. Neighborhood Shift (NEW)
# See how the neighbors change from royal words to baby words
df_neighborhood = analyzer.experiment_neighborhood_shift(TARGET_WORD, k=10)
display(df_neighborhood)

# 4. Ripple Effect
# Check if related words moved
neighbors = ["queen", "prince", "castle", "crown", "man"]
df_ripple = analyzer.experiment_ripple_effect(neighbors)
display(df_ripple)

# 5. Contextual Drift
analyzer.experiment_contextual_drift("The king is crying.", TARGET_WORD)

# 6. Visualization
analyzer.visualize(TARGET_WORD, CONCEPT_SOURCE, ["queen", "milk", "toy"], OUTPUT_DIR / "drift_plot.png")

In [ ]:
analyzer.generate_comparison("The girl told mum that she wanted", max_length=30)

# Analysis: How much fine-tuning is needed to induce significant semantic drift?
- What concepts experience the most drift? --> assess highly co-occuring vs. completely unrelated words under manipulation
- How many samples/epochs are needed to see measurable drift?
- Relevance: strategic manipulation of model knowledge through data pollution --> e.g., governments injecting biased data to shift public opinion via LLMs